In [13]:
# Instalar bibliotecas necessárias
# No Databricks use: %pip install azure-storage-blob pandas pyarrow python-dotenv
# No Jupyter local use: !pip install azure-storage-blob pandas pyarrow python-dotenv

%pip install azure-storage-blob pandas pyarrow python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
import random
import datetime
from dotenv import load_dotenv

# Carregar variáveis do .env (para ambiente local)
load_dotenv()

# Tenta usar dbutils (Databricks). Se não existir, cai para variável de ambiente
try:
    connection_string = dbutils.secrets.get(scope="kvfiaptechprod", key="AZURE-STORAGE-CONNECTION")
except NameError:
    connection_string = os.getenv("AZURE_STORAGE_CONNECTION")

# Verifica se conseguiu recuperar
if not connection_string:
    raise ValueError("❌ Connection string não encontrada. Configure no Key Vault (Databricks) ou no arquivo .env local.")

# Nome do container bronze
container_name = "bronze"

# Criar cliente de serviço
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
print("✅ Conexão com Azure Storage estabelecida")


✅ Conexão com Azure Storage estabelecida


In [15]:
# Função que gera um registro fictício baseado na tabela INEP Alunos
def gerar_registro():
    return {
        "ano": random.choice([2023, 2024]),
        "id_municipio": str(random.randint(1000000, 9999999)),
        "id_escola": f"E{random.randint(1000,9999)}",
        "id_aluno": f"A{random.randint(100000,999999)}",
        "proficiencia": round(random.uniform(100, 300), 2),
        "data_ingestao": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }


In [16]:
# Gerar um lote de 100 registros simulados
df = pd.DataFrame([gerar_registro() for _ in range(100)])

# Visualizar os primeiros registros
print("👀 Visualização dos dados simulados:")
display(df.head())

# Converter para Parquet em memória
parquet_buffer = io.BytesIO()
df.to_parquet(parquet_buffer, index=False, engine="pyarrow")

# Nome do arquivo com partição por data
data_particao = datetime.datetime.now().strftime("%Y/%m/%d")
blob_name = f"inep_alunos_simulado/{data_particao}/dados.parquet"

# Upload para o container bronze
blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)

print(f"✅ Dados simulados gravados no bronze em {blob_name}")


👀 Visualização dos dados simulados:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,8352368,E6937,A852930,147.07,2026-08-24 19:11:25
1,2023,6334500,E8310,A862748,280.72,2026-08-24 19:11:25
2,2023,7353152,E4188,A228724,267.83,2026-08-24 19:11:25
3,2024,3292905,E9195,A529441,251.51,2026-08-24 19:11:25
4,2023,5214212,E4156,A763853,113.65,2026-08-24 19:11:25


✅ Dados simulados gravados no bronze em inep_alunos_simulado/2026/08/24/dados.parquet


In [17]:
import time

# Simular streaming: gerar e gravar novos dados a cada 5 segundos
for i in range(5):  # número de ciclos
    df = pd.DataFrame([gerar_registro() for _ in range(10)])  # 10 registros por ciclo
    
    # Visualizar os primeiros registros de cada lote
    print(f"👀 Lote {i+1} - preview:")
    display(df.head())
    
    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False, engine="pyarrow")
    
    data_particao = datetime.datetime.now().strftime("%Y/%m/%d/%H%M%S")
    blob_name = f"inep_alunos_streaming/{data_particao}/dados.parquet"
    
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    blob_client.upload_blob(parquet_buffer.getvalue(), overwrite=True)
    
    print(f"📤 Lote {i+1} enviado para {blob_name}")
    time.sleep(5)  # espera 5 segundos antes do próximo lote


👀 Lote 1 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,3300503,E6211,A781470,159.37,2026-08-24 19:11:26
1,2023,8446374,E4002,A567544,133.36,2026-08-24 19:11:26
2,2023,5235982,E4058,A619998,286.28,2026-08-24 19:11:26
3,2024,1973388,E2420,A773443,250.45,2026-08-24 19:11:26
4,2024,1822889,E6167,A608300,255.19,2026-08-24 19:11:26


📤 Lote 1 enviado para inep_alunos_streaming/2026/08/24/191126/dados.parquet


👀 Lote 2 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,2791031,E1385,A160059,246.00,2026-08-24 19:11:31
1,2024,8866417,E4151,A488624,154.31,2026-08-24 19:11:31
2,2023,1998766,E7772,A935283,272.92,2026-08-24 19:11:31
3,2024,9629258,E9347,A225855,192.98,2026-08-24 19:11:31
4,2023,9890944,E6760,A226898,276.02,2026-08-24 19:11:31


...

📤 Lote 3 enviado para inep_alunos_streaming/2026/08/24/191139/dados.parquet
👀 Lote 4 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,1548923,E2029,A980572,240.92,2026-08-24 19:11:44
1,2023,5152875,E7729,A452163,177.52,2026-08-24 19:11:44
2,2023,3351591,E6180,A153222,267.72,2026-08-24 19:11:44
3,2024,2698105,E9761,A729529,155.72,2026-08-24 19:11:44
4,2023,5308614,E3455,A168555,181.86,2026-08-24 19:11:44


📤 Lote 4 enviado para inep_alunos_streaming/2026/08/24/191144/dados.parquet
👀 Lote 5 - preview:


,ano,id_municipio,id_escola,id_aluno,proficiencia,data_ingestao
0,2023,2796559,E9459,A562631,165.39,2026-08-24 19:11:49
1,2023,5650664,E3466,A221770,220.79,2026-08-24 19:11:49
2,2024,7885829,E1856,A422280,180.91,2026-08-24 19:11:49
3,2023,5800642,E7371,A893276,173.20,2026-08-24 19:11:49
4,2023,1100781,E5384,A914466,228.26,2026-08-24 19:11:49


📤 Lote 5 enviado para inep_alunos_streaming/2026/08/24/191149/dados.parquet
